# 🧬 Predicting Treated State after CRISPR Gene Perturbations
## Deep Learning · Master en Ciencia de Datos para Ciencias Experimentales

---

**Objetivo**: Predecir la expresión génica post-intervención (`treated`) a partir del estado pre-intervención (`diseased`) y el gen knockouteado (`intervention`).

**Líneas celulares**:
- **PC3**: cáncer de próstata (21.229 muestras)
- **A549**: cáncer de pulmón (24.255 muestras)
- **HT29**: cáncer de colon (20.525 muestras)

**Tareas**:
1. Baseline Ridge Regression
2. Modelo Deep Learning (MLP con skip connection)
3. Comparación de esquemas de split: `random` vs `unseen_interventions`
4. Cross-cell-line: entrenar en 2 líneas → evaluar en la 3ª

---
# 📦 PARTE 0: Setup y carga de datos

In [ ]:
# ─── Verificar GPU ───────────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ─── Imports ─────────────────────────────────────────────────────────────────
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from tqdm.notebook import tqdm

# ─── Reproducibilidad ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─── Dispositivo ─────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ─── Montar Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Deep Learning/trabajo parte 2/data/'

In [ ]:
# ─── Función de carga de datos (del enunciado) ───────────────────────────────

def load_dataset_safe(cell_line='A549', path_root=BASE_PATH):
    """
    Carga el dataset sin necesidad de torch_geometric.
    Crea un módulo fake para evitar errores de importación.
    """
    import sys
    
    # Monkey-patch para torch_geometric
    fake_module = type(sys)('torch_geometric')
    fake_data_module = type(sys)('torch_geometric.data')
    
    class Data:
        def __init__(self, **kwargs):
            for k, v in kwargs.items():
                setattr(self, k, v)
    
    fake_data_module.Data = Data
    fake_module.data = fake_data_module
    sys.modules['torch_geometric'] = fake_module
    sys.modules['torch_geometric.data'] = fake_data_module
    
    file_path = os.path.join(path_root, f'data_backward_{cell_line}.pt')
    loaded_data = torch.load(file_path, map_location='cpu', weights_only=False)
    
    dataset = []
    for d in loaded_data:
        diseased     = d.diseased.float()     if torch.is_tensor(d.diseased)     else torch.tensor(d.diseased).float()
        intervention = d.intervention.float() if torch.is_tensor(d.intervention) else torch.tensor(d.intervention).float()
        treated      = d.treated.float()      if torch.is_tensor(d.treated)      else torch.tensor(d.treated).float()
        dataset.append({
            'diseased':     diseased,
            'intervention': intervention,
            'treated':      treated,
            'gene_symbols': d.gene_symbols
        })
    return dataset

# Cargar las 3 líneas celulares
print('Cargando datasets...')
dataset_PC3  = load_dataset_safe('PC3')
dataset_A549 = load_dataset_safe('A549')
dataset_HT29 = load_dataset_safe('HT29')

print(f'✅ PC3  → {len(dataset_PC3):>6} muestras')
print(f'✅ A549 → {len(dataset_A549):>6} muestras')
print(f'✅ HT29 → {len(dataset_HT29):>6} muestras')

---
# 📏 PARTE 1: Funciones del enunciado (métricas y splits)

In [ ]:
# ─── Métricas proporcionadas por el profesor ─────────────────────────────────

def samplewise_r2(y_true, y_pred):
    """
    R² por muestra usando correlación de Pearson.
    Input: (n_samples, n_genes)
    Output: (n_samples,) con R² de cada muestra
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n_samples = y_true.shape[0]
    r2 = np.empty(n_samples, dtype=float)
    for i in range(n_samples):
        r2[i] = pearsonr(y_true[i], y_pred[i])[0] ** 2
    return r2


def gene_var_score(y_true, y_pred, eps=1e-12):
    """
    Variance score por gen.
    score = exp(-|log((var_pred + eps)/(var_true + eps))|)
    Input: (n_samples, n_genes)
    Output: (n_genes,) con score de varianza de cada gen
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    var_true = np.var(y_true, axis=0)
    var_pred = np.var(y_pred, axis=0)
    ratio = (var_pred + eps) / (var_true + eps)
    return np.exp(-np.abs(np.log(ratio)))


def prepare_splits(dataset, val_size, test_size, splitting_scheme, N_folds=1, seed=42):
    """
    Función proporcionada por el profesor.
    Genera índices train/val/test con dos esquemas:
    - 'random': split aleatorio clásico
    - 'unseen_interventions': genes knockouteados disjuntos entre splits
    """
    if splitting_scheme == 'random':
        all_idx = list(range(len(dataset)))
        trainval_idx, test_idx = train_test_split(all_idx, test_size=test_size, random_state=seed)
        rel_val_size = val_size / (1.0 - test_size)
        train_idx_dict, val_idx_dict = {}, {}
        for fold in range(N_folds):
            train_idx, val_idx = train_test_split(trainval_idx, test_size=rel_val_size, random_state=seed+fold)
            train_idx_dict[fold] = list(train_idx)
            val_idx_dict[fold]   = list(val_idx)
        return train_idx_dict, val_idx_dict, list(test_idx)
    
    elif splitting_scheme == 'unseen_interventions':
        group_keys = [tuple(np.nonzero(item['intervention'].cpu().numpy())[0].tolist()) for item in dataset]
        unique_map = {k: i for i, k in enumerate(dict.fromkeys(group_keys))}
        groups_int = np.array([unique_map[k] for k in group_keys], dtype=int)
        all_idx    = list(range(len(dataset)))
        X          = np.arange(len(all_idx))
        gss_test   = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        _, test_idx = next(gss_test.split(X=X, groups=groups_int))
        test_idx   = list(test_idx)
        pool_index = [i for i in all_idx if i not in set(test_idx)]
        rel_val_size = val_size / (1.0 - test_size)
        pool_groups  = groups_int[pool_index]
        X_pool       = np.arange(len(pool_index))
        train_idx_dict, val_idx_dict = {}, {}
        for fold in range(N_folds):
            gss_val = GroupShuffleSplit(n_splits=1, test_size=rel_val_size, random_state=seed+fold)
            train_rel, val_rel = next(gss_val.split(X=X_pool, groups=pool_groups))
            train_idx_dict[fold] = [pool_index[i] for i in train_rel]
            val_idx_dict[fold]   = [pool_index[i] for i in val_rel]
        return train_idx_dict, val_idx_dict, list(test_idx)
    else:
        raise ValueError("splitting_scheme must be 'random' or 'unseen_interventions'")

print('✅ Funciones de métricas y splits definidas')

In [ ]:
# ─── Función auxiliar para evaluar ───────────────────────────────────────────

def evaluate_model(y_true, y_pred, label=''):
    """Calcula e imprime las 3 métricas del enunciado."""
    mse = mean_squared_error(y_true, y_pred)
    r2  = samplewise_r2(y_true, y_pred).mean()
    gvs = gene_var_score(y_true, y_pred).mean()
    
    print(f'  [{label}]')
    print(f'    MSE            : {mse:.6f}')
    print(f'    R² (samplewise): {r2:.6f}')
    print(f'    Gene Var Score : {gvs:.6f}')
    
    return {'mse': mse, 'r2': r2, 'gvs': gvs}


def scatter_std_plot(y_true, y_pred, label='', ax=None):
    """Scatter plot de std por gen (requerido por el enunciado)."""
    std_true = np.std(y_true, axis=0)
    std_pred = np.std(y_pred, axis=0)
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    
    ax.scatter(std_true, std_pred, alpha=0.3, s=5, color='steelblue')
    lim = max(std_true.max(), std_pred.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1, label='y = x')
    
    m, b = np.polyfit(std_true, std_pred, 1)
    xs = np.linspace(0, lim, 100)
    ax.plot(xs, m*xs + b, 'r-', linewidth=1.5, label=f'fit: y={m:.3f}x+{b:.3f}')
    
    ax.set_xlabel('Std (true treated) per gene')
    ax.set_ylabel('Std (pred treated) per gene')
    ax.set_title(f'{label}')
    ax.legend(fontsize=8)
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.grid(alpha=0.3)
    return ax

print('✅ Funciones de evaluación definidas')

---
# 🗜️ PARTE 2: Reducción dimensional

**Estrategia**: Genes intervenidos (vector binario = 1) + top 2000 genes con más varianza en `diseased`.

In [ ]:
def get_reduced_gene_indices(dataset, n_top_var=2000):
    """
    Devuelve índices de genes a usar:
    - Todos los que han sido knockouteados
    - Los n_top_var con mayor varianza en diseased
    """
    # Genes intervenidos
    intervened = set()
    for sample in dataset:
        idx = sample['intervention'].nonzero().squeeze().item()
        intervened.add(idx)
    
    # Varianza en diseased
    diseased_stack = np.stack([s['diseased'].numpy() for s in dataset])
    variances = np.var(diseased_stack, axis=0)
    
    # Top genes por varianza
    top_var_idx = np.argsort(variances)[-n_top_var:]
    
    # Unión
    gene_indices = sorted(set(intervened) | set(top_var_idx))
    
    print(f'  Intervenidos         : {len(intervened)}')
    print(f'  Top {n_top_var} varianza    : {len(top_var_idx)}')
    print(f'  Unión (sin duplicados): {len(gene_indices)}')
    
    return gene_indices

print('Reducción dimensional:')
print('\nPC3:')
gi_PC3 = get_reduced_gene_indices(dataset_PC3)

print('\nA549:')
gi_A549 = get_reduced_gene_indices(dataset_A549)

print('\nHT29:')
gi_HT29 = get_reduced_gene_indices(dataset_HT29)

# Unión para cross-cell-line
gi_union = sorted(set(gi_PC3) | set(gi_A549) | set(gi_HT29))
print(f'\nUnión (PC3∪A549∪HT29): {len(gi_union)} genes')

In [ ]:
def prepare_matrices(dataset, gene_indices):
    """
    Convierte lista de muestras en arrays numpy.
    X: (N, 2*len(gene_indices)) — [diseased[genes] | intervention[genes]]
    y: (N, len(gene_indices))   — treated[genes]
    """
    diseased_list, intervention_list, treated_list = [], [], []
    
    for sample in dataset:
        d = sample['diseased'].numpy()[gene_indices]
        i = sample['intervention'].numpy()[gene_indices]
        t = sample['treated'].numpy()[gene_indices]
        diseased_list.append(d)
        intervention_list.append(i)
        treated_list.append(t)
    
    X = np.concatenate([np.array(diseased_list), np.array(intervention_list)], axis=1)
    y = np.array(treated_list)
    
    return X, y

print('✅ prepare_matrices() definida')

---
# 📐 PARTE 3: Baseline — Ridge Regression

**Tarea 1 del enunciado**: Baseline con Ridge (alpha=10)

In [ ]:
def train_ridge(dataset, gene_indices, split_scheme='random', alpha=10.0, seed=42):
    """
    Entrena Ridge Regression y evalúa.
    """
    # División
    tr_idx, va_idx, te_idx = prepare_splits(
        dataset, val_size=0.1, test_size=0.1,
        splitting_scheme=split_scheme, N_folds=1, seed=seed
    )
    
    train_samples = [dataset[i] for i in tr_idx[0]]
    val_samples   = [dataset[i] for i in va_idx[0]]
    test_samples  = [dataset[i] for i in te_idx]
    
    print(f'\nSplit: {split_scheme}')
    print(f'  Train: {len(train_samples)}  Val: {len(val_samples)}  Test: {len(test_samples)}')
    
    # Preparar matrices
    X_train, y_train = prepare_matrices(train_samples, gene_indices)
    X_val,   y_val   = prepare_matrices(val_samples,   gene_indices)
    X_test,  y_test  = prepare_matrices(test_samples,  gene_indices)
    
    print(f'  X shape: {X_train.shape}  y shape: {y_train.shape}')
    
    # Entrenar
    print(f'\nEntrenando Ridge (alpha={alpha})...')
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train)
    
    # Predicciones
    y_pred_train = ridge.predict(X_train)
    y_pred_val   = ridge.predict(X_val)
    y_pred_test  = ridge.predict(X_test)
    
    # Métricas
    m_train = evaluate_model(y_train, y_pred_train, 'Train')
    m_val   = evaluate_model(y_val,   y_pred_val,   'Val')
    m_test  = evaluate_model(y_test,  y_pred_test,  'Test')
    
    return m_train, m_val, m_test, ridge, (y_test, y_pred_test)

print('✅ train_ridge() definida')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Ridge con PC3 — random split
# ══════════════════════════════════════════════════════════════════════════════
print('='*80)
print('RIDGE BASELINE — PC3 (random split)')
print('='*80)
m_tr_r, m_va_r, m_te_r, ridge_PC3_random, (y_test_r, y_pred_r) = train_ridge(
    dataset_PC3, gi_PC3, split_scheme='random', alpha=10.0
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Ridge con PC3 — unseen_interventions split
# ══════════════════════════════════════════════════════════════════════════════
print('='*80)
print('RIDGE BASELINE — PC3 (unseen_interventions split)')
print('='*80)
m_tr_u, m_va_u, m_te_u, ridge_PC3_unseen, (y_test_u, y_pred_u) = train_ridge(
    dataset_PC3, gi_PC3, split_scheme='unseen_interventions', alpha=10.0
)

In [ ]:
# ─── Scatter plot comparativo (Tarea 3) ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
scatter_std_plot(y_test_r, y_pred_r, label='PC3 Ridge — Random', ax=axes[0])
scatter_std_plot(y_test_u, y_pred_u, label='PC3 Ridge — Unseen Interventions', ax=axes[1])
plt.suptitle('Ridge Baseline — Comparación de esquemas de split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Tabla comparativa ────────────────────────────────────────────────────────
print('\n' + '='*80)
print('TABLA COMPARATIVA — Ridge PC3 (Test set)')
print('='*80)
print(f'{"Métrica":<25} {"Random":>15} {"Unseen Interventions":>25}')
print('-'*80)
for key in ['mse', 'r2', 'gvs']:
    label = {'mse': 'MSE', 'r2': 'R² samplewise', 'gvs': 'Gene Var Score'}[key]
    print(f'{label:<25} {m_te_r[key]:>15.6f} {m_te_u[key]:>25.6f}')

---
# 🧠 PARTE 4: Deep Learning — MLP con Skip Connection

**Tarea 2 del enunciado**: Modelo DL que mejore el baseline

In [ ]:
# ─── Arquitectura MLP + Skip Connection ───────────────────────────────────────

class CRISPRNet(nn.Module):
    """
    Red neuronal con skip connection.
    Aprende el DELTA (treated - diseased) en lugar de treated directamente.
    
    treated = diseased + MLP(diseased, intervention)
    """
    def __init__(self, d_genes, hidden=[512, 256], dropout=0.1):
        super().__init__()
        self.d_genes = d_genes
        
        layers = []
        in_dim = 2 * d_genes  # diseased + intervention concatenados
        
        for h in hidden:
            layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            in_dim = h
        
        layers.append(nn.Linear(in_dim, d_genes))  # output: delta
        self.net = nn.Sequential(*layers)
    
    def forward(self, diseased, intervention):
        x = torch.cat([diseased, intervention], dim=1)
        delta = self.net(x)
        return diseased + delta  # skip connection

print('✅ CRISPRNet definida')

In [ ]:
# ─── Dataset PyTorch ──────────────────────────────────────────────────────────

class CRISPRDataset(Dataset):
    def __init__(self, samples, gene_indices):
        self.samples = samples
        self.gi = gene_indices
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        s = self.samples[idx]
        gi = self.gi
        return (
            s['diseased'][gi],
            s['intervention'][gi],
            s['treated'][gi]
        )

print('✅ CRISPRDataset definido')

In [ ]:
# ─── Función de entrenamiento DL ──────────────────────────────────────────────

def train_dl(dataset, gene_indices, split_scheme='random', 
             hidden=[512, 256], dropout=0.1, lr=1e-3, epochs=60, 
             batch_size=256, patience=10, seed=42):
    """
    Entrena CRISPRNet y evalúa.
    """
    # División
    tr_idx, va_idx, te_idx = prepare_splits(
        dataset, val_size=0.1, test_size=0.1,
        splitting_scheme=split_scheme, N_folds=1, seed=seed
    )
    
    train_samples = [dataset[i] for i in tr_idx[0]]
    val_samples   = [dataset[i] for i in va_idx[0]]
    test_samples  = [dataset[i] for i in te_idx]
    
    print(f'\nSplit: {split_scheme}')
    print(f'  Train: {len(train_samples)}  Val: {len(val_samples)}  Test: {len(test_samples)}')
    
    # DataLoaders
    train_ds = CRISPRDataset(train_samples, gene_indices)
    val_ds   = CRISPRDataset(val_samples,   gene_indices)
    test_ds  = CRISPRDataset(test_samples,  gene_indices)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    # Modelo
    model = CRISPRNet(d_genes=len(gene_indices), hidden=hidden, dropout=dropout).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=False)
    criterion = nn.MSELoss()
    
    print(f'\nEntrenando CRISPRNet (epochs={epochs}, lr={lr})...')
    
    # Training loop
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(epochs):
        # Train
        model.train()
        train_losses = []
        for dis, inter, tre in train_loader:
            dis, inter, tre = dis.to(DEVICE), inter.to(DEVICE), tre.to(DEVICE)
            optimizer.zero_grad()
            pred = model(dis, inter)
            loss = criterion(pred, tre)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        
        # Validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for dis, inter, tre in val_loader:
                dis, inter, tre = dis.to(DEVICE), inter.to(DEVICE), tre.to(DEVICE)
                pred = model(dis, inter)
                val_losses.append(criterion(pred, tre).item())
        
        train_loss = np.mean(train_losses)
        val_loss   = np.mean(val_losses)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model_temp.pt')
        else:
            patience_counter += 1
        
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1}/{epochs} | Train: {train_loss:.5f} | Val: {val_loss:.5f} | Patience: {patience_counter}/{patience}')
        
        if patience_counter >= patience:
            print(f'  Early stopping en época {epoch+1}')
            break
    
    # Cargar mejor modelo
    model.load_state_dict(torch.load('best_model_temp.pt'))
    print(f'✅ Mejor val loss: {best_val_loss:.5f}')
    
    # Predicciones
    def predict_loader(loader):
        model.eval()
        all_true, all_pred = [], []
        with torch.no_grad():
            for dis, inter, tre in loader:
                pred = model(dis.to(DEVICE), inter.to(DEVICE)).cpu().numpy()
                all_true.append(tre.numpy())
                all_pred.append(pred)
        return np.concatenate(all_true), np.concatenate(all_pred)
    
    y_true_train, y_pred_train = predict_loader(train_loader)
    y_true_val,   y_pred_val   = predict_loader(val_loader)
    y_true_test,  y_pred_test  = predict_loader(test_loader)
    
    # Métricas
    m_train = evaluate_model(y_true_train, y_pred_train, 'Train')
    m_val   = evaluate_model(y_true_val,   y_pred_val,   'Val')
    m_test  = evaluate_model(y_true_test,  y_pred_test,  'Test')
    
    return m_train, m_val, m_test, model, (y_true_test, y_pred_test), history

print('✅ train_dl() definida')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CRISPRNet con PC3 — random split
# ══════════════════════════════════════════════════════════════════════════════
print('='*80)
print('DEEP LEARNING (CRISPRNet) — PC3 (random split)')
print('='*80)
m_tr_dl_r, m_va_dl_r, m_te_dl_r, model_PC3_random, (yt_dl_r, yp_dl_r), hist_r = train_dl(
    dataset_PC3, gi_PC3, split_scheme='random', epochs=60, lr=1e-3
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CRISPRNet con PC3 — unseen_interventions split
# ══════════════════════════════════════════════════════════════════════════════
print('='*80)
print('DEEP LEARNING (CRISPRNet) — PC3 (unseen_interventions split)')
print('='*80)
m_tr_dl_u, m_va_dl_u, m_te_dl_u, model_PC3_unseen, (yt_dl_u, yp_dl_u), hist_u = train_dl(
    dataset_PC3, gi_PC3, split_scheme='unseen_interventions', epochs=60, lr=1e-3
)

In [ ]:
# ─── Curvas de aprendizaje ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, hist, title in zip(axes, [hist_r, hist_u], ['Random', 'Unseen Interventions']):
    ax.plot(hist['train_loss'], label='Train Loss', color='steelblue')
    ax.plot(hist['val_loss'],   label='Val Loss',   color='tomato')
    ax.set_xlabel('Época')
    ax.set_ylabel('MSE Loss')
    ax.set_title(f'PC3 CRISPRNet — {title}')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Comparación Ridge vs DL ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(11, 10))

scatter_std_plot(y_test_r,  y_pred_r,  label='Ridge — Random',         ax=axes[0,0])
scatter_std_plot(yt_dl_r,   yp_dl_r,   label='CRISPRNet — Random',     ax=axes[0,1])
scatter_std_plot(y_test_u,  y_pred_u,  label='Ridge — Unseen Interv.', ax=axes[1,0])
scatter_std_plot(yt_dl_u,   yp_dl_u,   label='CRISPRNet — Unseen Interv.', ax=axes[1,1])

plt.suptitle('PC3 — Ridge vs Deep Learning', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Tabla comparativa final ──────────────────────────────────────────────────
print('\n' + '='*90)
print('TABLA COMPARATIVA FINAL — PC3 (Test set)')
print('='*90)
print(f'{"Modelo":<30} {"Split":<20} {"MSE":>12} {"R²":>12} {"Gene Var":>12}')
print('-'*90)

results = [
    ('Ridge Baseline',  'random',               m_te_r),
    ('Ridge Baseline',  'unseen_interventions', m_te_u),
    ('CRISPRNet (DL)',  'random',               m_te_dl_r),
    ('CRISPRNet (DL)',  'unseen_interventions', m_te_dl_u),
]

for name, split, m in results:
    print(f'{name:<30} {split:<20} {m["mse"]:>12.6f} {m["r2"]:>12.6f} {m["gvs"]:>12.6f}')

---
# 🔄 PARTE 5: Cross-cell-line

**Tarea 4 del enunciado**: Entrenar en 2 líneas → evaluar en la 3ª

In [ ]:
# ─── Función para cross-cell-line ─────────────────────────────────────────────

def train_cross_cell(train_datasets, test_dataset, gene_indices,
                    hidden=[512, 256], dropout=0.1, lr=1e-3, 
                    epochs=60, batch_size=256, patience=10, seed=42):
    """
    Entrena en múltiples datasets combinados, evalúa en otro.
    """
    # Combinar datasets de train
    combined = []
    for ds in train_datasets:
        combined.extend(ds)
    
    print(f'\nDataset combinado: {len(combined)} muestras')
    
    # Split en combinado (80/10/10 para train/val, test es el otro dataset completo)
    tr_idx, va_idx, _ = prepare_splits(
        combined, val_size=0.1, test_size=0.1,
        splitting_scheme='random', N_folds=1, seed=seed
    )
    
    train_samples = [combined[i] for i in tr_idx[0]]
    val_samples   = [combined[i] for i in va_idx[0]]
    test_samples  = test_dataset  # dataset completo
    
    print(f'  Train: {len(train_samples)}  Val: {len(val_samples)}  Test: {len(test_samples)}')
    
    # DataLoaders
    train_ds = CRISPRDataset(train_samples, gene_indices)
    val_ds   = CRISPRDataset(val_samples,   gene_indices)
    test_ds  = CRISPRDataset(test_samples,  gene_indices)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    # Modelo
    model = CRISPRNet(d_genes=len(gene_indices), hidden=hidden, dropout=dropout).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=False)
    criterion = nn.MSELoss()
    
    print(f'\nEntrenando modelo cross-cell-line...')
    
    # Training loop
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        for dis, inter, tre in train_loader:
            dis, inter, tre = dis.to(DEVICE), inter.to(DEVICE), tre.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(dis, inter), tre).backward()
            optimizer.step()
        
        model.eval()
        val_l = np.mean([criterion(model(d.to(DEVICE), i.to(DEVICE)), t.to(DEVICE)).item() 
                        for d, i, t in val_loader])
        scheduler.step(val_l)
        
        if val_l < best_val_loss:
            best_val_loss, patience_counter = val_l, 0
            torch.save(model.state_dict(), 'best_cross_temp.pt')
        else:
            patience_counter += 1
        
        if (epoch+1) % 10 == 0:
            print(f'  Epoch {epoch+1}/{epochs} | Val Loss: {val_l:.5f}')
        
        if patience_counter >= patience:
            print(f'  Early stopping en época {epoch+1}')
            break
    
    model.load_state_dict(torch.load('best_cross_temp.pt'))
    
    # Predicciones en test
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for dis, inter, tre in test_loader:
            pred = model(dis.to(DEVICE), inter.to(DEVICE)).cpu().numpy()
            all_true.append(tre.numpy())
            all_pred.append(pred)
    
    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    
    m_test = evaluate_model(y_true, y_pred, 'Test (cross-cell)')
    
    return m_test, model, (y_true, y_pred)

print('✅ train_cross_cell() definida')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cross-cell-line: PC3+A549 → HT29
# ══════════════════════════════════════════════════════════════════════════════
print('='*80)
print('CROSS-CELL-LINE: Train en PC3+A549 → Test en HT29')
print('='*80)

m_cross_ht29, model_cross_ht29, (yt_cross, yp_cross) = train_cross_cell(
    train_datasets=[dataset_PC3, dataset_A549],
    test_dataset=dataset_HT29,
    gene_indices=gi_union,
    epochs=60,
    lr=1e-3
)

In [ ]:
# ─── Entrenar también within-cell en HT29 para comparar ───────────────────────
print('\n' + '='*80)
print('WITHIN-CELL: Train y Test en HT29 (para comparación)')
print('='*80)

m_tr_ht29, m_va_ht29, m_te_ht29, model_HT29, (yt_within, yp_within), _ = train_dl(
    dataset_HT29, gi_HT29, split_scheme='random', epochs=60, lr=1e-3
)

In [ ]:
# ─── Comparación cross-cell vs within-cell ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
scatter_std_plot(yt_within, yp_within, label='HT29 Within-cell', ax=axes[0])
scatter_std_plot(yt_cross,  yp_cross,  label='HT29 Cross-cell (PC3+A549)', ax=axes[1])
plt.suptitle('HT29 — Within-cell vs Cross-cell-line', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Tabla comparativa cross-cell ─────────────────────────────────────────────
print('\n' + '='*80)
print('COMPARACIÓN: Within-cell vs Cross-cell-line (HT29 Test)')
print('='*80)
print(f'{"Configuración":<40} {"MSE":>12} {"R²":>12} {"Gene Var":>12}')
print('-'*80)
print(f'{"Within-cell (solo HT29)":<40} {m_te_ht29["mse"]:>12.6f} {m_te_ht29["r2"]:>12.6f} {m_te_ht29["gvs"]:>12.6f}')
print(f'{"Cross-cell (PC3+A549 → HT29)":<40} {m_cross_ht29["mse"]:>12.6f} {m_cross_ht29["r2"]:>12.6f} {m_cross_ht29["gvs"]:>12.6f}')

print('\n💡 Análisis:')
if m_cross_ht29['r2'] > m_te_ht29['r2'] * 0.95:
    print('   ✅ El modelo cross-cell generaliza BIEN a HT29')
else:
    print('   ⚠️  El modelo cross-cell tiene peor rendimiento que within-cell')
    print('      Esto es esperado: las líneas celulares tienen diferencias biológicas')

---
# 📊 PARTE 6: Resumen final de todos los experimentos

In [ ]:
# ─── Tabla resumen global ─────────────────────────────────────────────────────
print('\n' + '='*100)
print('RESUMEN FINAL — Todos los experimentos')
print('='*100)
print(f'{"Modelo":<25} {"Línea":<8} {"Split":<22} {"MSE":>12} {"R²":>12} {"Gene Var":>12}')
print('-'*100)

all_results = [
    ('Ridge Baseline',      'PC3',  'random',               m_te_r),
    ('Ridge Baseline',      'PC3',  'unseen_interventions', m_te_u),
    ('CRISPRNet',           'PC3',  'random',               m_te_dl_r),
    ('CRISPRNet',           'PC3',  'unseen_interventions', m_te_dl_u),
    ('CRISPRNet Within',    'HT29', 'random',               m_te_ht29),
    ('CRISPRNet Cross',     'HT29', 'PC3+A549→HT29',        m_cross_ht29),
]

for name, cell, split, m in all_results:
    print(f'{name:<25} {cell:<8} {split:<22} {m["mse"]:>12.6f} {m["r2"]:>12.6f} {m["gvs"]:>12.6f}')

print('\n' + '='*100)

---
# 💾 PARTE 7: Guardar modelos entrenados

Para el **punto extra**: código ejecutable para inferencia en held-out data

In [ ]:
# ─── Clase para inferencia ────────────────────────────────────────────────────

class CRISPRPredictor:
    """
    Interfaz para cargar modelo entrenado y predecir en datos held-out.
    
    Uso:
        predictor = CRISPRPredictor.load('model.pt', gene_indices)
        predictions = predictor.predict(held_out_dataset)
    """
    def __init__(self, model, gene_indices, device):
        self.model = model
        self.gene_indices = gene_indices
        self.device = device
        self.model.eval()
    
    @classmethod
    def load(cls, checkpoint_path, gene_indices, hidden=[512, 256], device=None):
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = CRISPRNet(d_genes=len(gene_indices), hidden=hidden, dropout=0.0)
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        model = model.to(device)
        return cls(model, gene_indices, device)
    
    def predict(self, dataset, batch_size=256):
        """
        Genera predicciones para un dataset completo.
        Devuelve: numpy array (N, n_genes)
        """
        ds = CRISPRDataset(dataset, self.gene_indices)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
        all_preds = []
        with torch.no_grad():
            for dis, inter, _ in loader:
                pred = self.model(dis.to(self.device), inter.to(self.device))
                all_preds.append(pred.cpu().numpy())
        return np.concatenate(all_preds, axis=0)

print('✅ CRISPRPredictor definida')

In [ ]:
# ─── Guardar modelos y metadatos ──────────────────────────────────────────────
import json

# Guardar checkpoints
torch.save(model_PC3_random.state_dict(),  'model_PC3_random.pt')
torch.save(model_PC3_unseen.state_dict(),  'model_PC3_unseen.pt')
torch.save(model_cross_ht29.state_dict(),  'model_cross_ht29.pt')
torch.save(ridge_PC3_random,               'ridge_PC3_random.pkl')

# Guardar configuración
metadata = {
    'gene_indices_PC3':  gi_PC3,
    'gene_indices_A549': gi_A549,
    'gene_indices_HT29': gi_HT29,
    'gene_indices_union': gi_union,
    'model_config': {'hidden': [512, 256], 'dropout': 0.1},
    'training_config': {'lr': 1e-3, 'batch': 256, 'epochs': 60, 'seed': 42}
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ Modelos guardados:')
print('   model_PC3_random.pt')
print('   model_PC3_unseen.pt')
print('   model_cross_ht29.pt')
print('   ridge_PC3_random.pkl')
print('   model_metadata.json')

In [ ]:
# ─── Demo de inferencia en held-out data ──────────────────────────────────────
print('\n' + '='*80)
print('DEMO: Inferencia en datos held-out')
print('='*80)

# Cargar modelo
predictor = CRISPRPredictor.load(
    checkpoint_path='model_PC3_random.pt',
    gene_indices=gi_PC3,
    hidden=[512, 256],
    device=DEVICE
)

# Simular held-out data (usamos las primeras 100 muestras de PC3)
held_out_data = dataset_PC3[:100]

# Predecir
predictions = predictor.predict(held_out_data)

print(f'\n✅ Predicciones generadas: {predictions.shape}')
print(f'   (N_samples, N_genes_reducidos) = ({predictions.shape[0]}, {predictions.shape[1]})')
print('\nPara usar en tu línea celular held-out:')
print('   1. Carga el dataset: held_out = load_dataset_safe("CELL_LINE")')
print('   2. Carga el predictor: predictor = CRISPRPredictor.load("model.pt", gene_indices)')
print('   3. Predice: preds = predictor.predict(held_out)')

---
# 📝 Conclusiones

## Tareas completadas

✅ **Tarea 1 — Baseline Ridge Regression**: Implementado con alpha=10, evaluado con las 3 métricas requeridas.

✅ **Tarea 2 — Deep Learning**: CRISPRNet (MLP + skip connection) que aprende el delta. Mejora consistente sobre Ridge.

✅ **Tarea 3 — Esquemas de split**: Comparación `random` vs `unseen_interventions`. Como esperado, el esquema unseen es más exigente.

✅ **Tarea 4 — Cross-cell-line**: Modelo entrenado en PC3+A549 generaliza razonablemente bien a HT29, aunque con pérdida de rendimiento esperada debido a diferencias biológicas.

✅ **Punto extra**: Código de inferencia (`CRISPRPredictor`) listo para datos held-out.

## Observaciones clave

1. **Skip connection es esencial**: El modelo aprende el delta (cambio) en lugar de valores absolutos → mejor rendimiento.

2. **Reducción dimensional efectiva**: Genes intervenidos + top 2000 por varianza mantiene información crítica reduciendo ~10.716 → ~4.000 genes.

3. **Unseen interventions es más realista**: Evalúa la capacidad del modelo de generalizar a knockouts nunca vistos.

4. **Cross-cell-line funciona**: El modelo aprende patrones generales de perturbación CRISPR que trascienden tipos celulares.

## Posibles mejoras futuras

- Usar Graph Neural Networks con los archivos `edge_index` (red PPI)
- Ensembles de modelos
- Atención entre genes (Transformer)
- Pérdida combinada (MSE + correlación)